# 1. Environment Setup
Install the necessary Hugging Face and PyTorch libraries.

In [1]:
# Install required libraries
!pip install -q transformers datasets torch scikit-learn matplotlib seaborn


# 2. Imports & Data Preprocessing
This block handles data loading and basic cleaning as per the "Mandatory" requirement.


In [6]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import BertTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, load_dataset

# 1. DATA PREPROCESSING
# Using a sample of the IMDB dataset for faster execution
dataset = load_dataset('csv', data_files='/content/sample_data/IMDB Dataset.csv', split='train[:2000]')
df = pd.DataFrame(dataset)

# Basic cleaning: Remove missing values
df = df.dropna()

# 2. DATA SPLITTING (Train 70%, Val 15%, Test 15%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

Generating train split: 0 examples [00:00, ? examples/s]

# 3. Tokenization
Using the required bert-base-uncased tokenizer.

In [3]:
# 3. TOKENIZATION
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_func(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Convert to HuggingFace Dataset format
train_ds = Dataset.from_pandas(train_df).map(tokenize_func, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize_func, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_func, batched=True)



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

# 4. Experiments: Layer Freezing vs. Fine-Tuning
This block sets up the two experiments required by your documentation.

In [8]:
# Shared training settings
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1}

args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    eval_strategy='epoch'  # Changed from evaluation_strategy
)

# EXPERIMENT 1: Freeze all BERT layers
model_frozen = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
for param in model_frozen.bert.parameters():
    param.requires_grad = False

trainer_frozen = Trainer(model=model_frozen, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer_frozen.train()

# EXPERIMENT 2: Fine-tune last 2 layers
model_ft = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
for param in model_ft.bert.parameters():
    param.requires_grad = False
for layer in model_ft.bert.encoder.layer[-2:]: # Unfreeze last 2
    for param in layer.parameters():
        param.requires_grad = True

trainer_ft = Trainer(model=model_ft, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer_ft.train()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.170525,0.980000,0.000000
2,No log,0.107777,0.990000,0.000000


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# 5. Final Evaluation
Generate the Confusion Matrix and final metrics.

In [9]:
# Final Test Evaluation8
preds = trainer_ft.predict(test_ds)
y_preds = preds.predictions.argmax(-1)
y_true = test_df['label']

# Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix: Fine-tuned Model')
plt.show()
print("Final Test Metrics:", preds.metrics)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 